# Rotterdam CRM - Building Materials Analysis

**Author:** nataliaspalekgarcia  
**Created:** Mon Nov 10 17:07:42 2025

This notebook analyzes building material profiles, composite materials, and heat pump material intensities for the Rotterdam CRM project.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import os

import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Set working directory (update this path as needed)
# os.chdir(r"/Users/nataliaspalekgarcia/Desktop/25_80_CRM_Rotterdam/02_modelling")

## 2. Load Building Data

In [3]:
# Load building profiles and product materialisation data
data_building_profiles = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_building_profiles")

data_products_materialisation = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_products_materialisation")

## 3. Clean and Prepare Data

In [14]:
data_products_materialisation

,Bron,QC,Warnings,productcode,product,ElUnit,material_code,material_name,genericmaterialname,genericmaterialname_eng,...,Product-SBK Verwerking Check,SBK Oorsprong Check,Opmerkingen,Cat1/2/3,Lengte_m1,Breedte_m1,Dikte_m1,Volume_m3,%_mass_in_product,Rc waarde
0,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",47.04.020,DAK en MILIEU Bitumen gemod. tweelaags losligg...,m2,NaN,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,1.000000,NaN
1,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",31.14.005,Deurdrangers inclusief deur co-ordinators,stuks,NaN,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,1.000000,NaN
2,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",43.02.903,Entreemat,m2,NaN,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,1.000000,NaN
3,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",31.14.006,Hang- en sluitwerk voor schuifdeuren,stuks,NaN,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,1.000000,NaN
4,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",31.03.010H,Hergebruik - PVC op staalkern,NaN,NaN,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,1.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,NaN,NaN,"LET OP! Fouten gevonden, check Kolom P:R voor ...",61.04.903,"Energie laagspanning U-bouw, energie laagspann...",kWh,5000pr,Materialisatie energieopwekking (centrales obv...,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,NaN,NaN
1484,BENG Inputsheets,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",61.04.001,energie laagspanningsinstallatie inclusief ver...,kWh,762pr,laagspanning,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,NaN,NaN
1485,BENG Inputsheets,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",61.04.001,energie laagspanningsinstallatie inclusief ver...,kWh,5000pr,Materialisatie energieopwekking (centrales obv...,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,NaN,NaN
1486,NaN,ja,"LET OP! Fouten gevonden, check Kolom P:R voor ...",41.03.001,Pleisterwerk; geschilderd,m2,050,NaN,NaN,NaN,...,Product-SBK not in Verwerking Product-SBK sheet!,SBK not in Oorsprong SBK sheet!,NaN,NaN,NaN,NaN,NaN,Niet berekend,0.943262,NaN


In [4]:
# Select relevant columns from products materialisation
data_products_materialisation_clean = data_products_materialisation[[
    "productcode", "product", "material_code",
    "material_name", "genericmaterialname", "genericmaterialname_eng",
    "mass_kg", "S-laag"
]]

# Select relevant columns from building profiles
data_building_profiles_clean = data_building_profiles[[
    "Gebouwscenario", "Bron", "Projectnummer",
    "building_type", "m2bvo", "productcode", "Hoeveelheid", "units_per_m2"
]]

## 4. Merge and Filter Materials

In [5]:
# Merge building profiles with product materialisation
df_merged = pd.merge(
    data_building_profiles_clean,
    data_products_materialisation_clean,
    on="productcode",
    how="left"
)

# Keep only materials of interest
allowed_generic_names = ["Aluminium", "Beton", "Koper", "Staal & IJzer"]
df_filtered = df_merged[df_merged["genericmaterialname"].isin(allowed_generic_names)].copy()

In [6]:
df_merged

,Gebouwscenario,Bron,Projectnummer,building_type,m2bvo,productcode,Hoeveelheid,units_per_m2,product,material_code,material_name,genericmaterialname,genericmaterialname_eng,mass_kg,S-laag
0,BAU,Copper8,61.0,Appartement (> 6 lagen),51054.0,17.01.00016,12792.0,0.250558,"Schroefpaal; beton,in het werk gestort, C20/25...",844,Betonmortel C20/25 (o.b.v. 75% CEM III en 25% ...,Beton,Concrete,338.048780,Structure
1,BAU,Copper8,61.0,Appartement (> 6 lagen),51054.0,17.01.00016,12792.0,0.250558,"Schroefpaal; beton,in het werk gestort, C20/25...",767pr,Wapeningsstaal,Staal & IJzer,Steel,11.758218,Structure
2,BAU,Copper8,61.0,Appartement (> 6 lagen),51054.0,17.01.00016,12792.0,0.250558,"Schroefpaal; beton,in het werk gestort, C20/25...",767pr,Wapeningsstaal,Staal & IJzer,Steel,8.460000,Structure
3,BAU,Copper8,61.0,Appartement (> 6 lagen),51054.0,11.01.001,4026.0,0.078858,Zand,294,Zand (NVLB: B1 ophoogzand / B2 industriezand),Zand,Sand,160.000000,Site
4,BAU,Copper8,61.0,Appartement (> 6 lagen),51054.0,16.01.90012,384.0,0.007521,"Betonhuis; beton,in het werk gestort, C20/25,C...",844,Betonmortel C20/25 (o.b.v. 75% CEM III en 25% ...,Beton,Concrete,427.000000,Structure
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
777,BAU,Metabolic (bewerking van RVO referentiegebouwen),77.0,Appartement (<= 6 lagen),3828.0,73.01.001,79.2,0.020690,Multiplex; geschilderd:alkyd,163,"Multiplex, duurzame bosbouw",Hout,Wood,26.530000,Stuff
778,BAU,Metabolic (bewerking van RVO referentiegebouwen),77.0,Appartement (<= 6 lagen),3828.0,73.01.001,79.2,0.020690,Multiplex; geschilderd:alkyd,271,"Vuren, duurzame bosbouw",Hout,Wood,3.711000,Stuff
779,BAU,Metabolic (bewerking van RVO referentiegebouwen),77.0,Appartement (<= 6 lagen),3828.0,73.01.001,79.2,0.020690,Multiplex; geschilderd:alkyd,84,Hardboard,Hout,Wood,6.960000,Stuff
780,BAU,Metabolic (bewerking van RVO referentiegebouwen),77.0,Appartement (<= 6 lagen),3828.0,73.01.001,79.2,0.020690,Multiplex; geschilderd:alkyd,457,"Alkydharsverf, gemodificeerd, voor buiten (vol...",Lijm en verf,Paint and glue,0.364000,Stuff


## 5. Map Generic Materials to Composite Categories

In [9]:
# Map genericmaterialname to material_composite
generic_to_composite = {
    "Beton": "concrete",
    "Staal & IJzer": "steel",
    "Aluminium": "aluminium",
    "Koper": "copper"
}

df_filtered["material_composite"] = df_filtered["genericmaterialname"].map(generic_to_composite)

## 6. S-laag Analysis and Share Calculation

In [25]:
df_filtered['kg_per_m2'] = df_filtered['units_per_m2']*df_filtered['mass_kg']

In [26]:
# Aggregate total mass per material_composite × S-laag
df_slaag_composite_totals = (
    df_filtered
    .groupby(["material_composite", "S-laag"], as_index=False)
    ["kg_per_m2"]
    .sum()
)
df_slaag_composite_totals

,material_composite,S-laag,kg_per_m2
0,aluminium,Services,0.387110
1,aluminium,Skin,0.653695
2,aluminium,Space plan,0.446905
3,concrete,Services,0.004000
4,concrete,Skin,195.477583
5,concrete,Space plan,208.943653
6,concrete,Structure,2927.534693
7,copper,Services,0.811872
8,steel,Services,7.169211
9,steel,Skin,7.083634


In [27]:
# Total mass per material_composite
df_slaag_composite_totals["total_kg_material"] = (
    df_slaag_composite_totals
    .groupby("material_composite")["kg_per_m2"]
    .transform("sum")
)
df_slaag_composite_totals

,material_composite,S-laag,kg_per_m2,total_kg_material
0,aluminium,Services,0.387110,1.487710
1,aluminium,Skin,0.653695,1.487710
2,aluminium,Space plan,0.446905,1.487710
3,concrete,Services,0.004000,3331.959929
4,concrete,Skin,195.477583,3331.959929
5,concrete,Space plan,208.943653,3331.959929
6,concrete,Structure,2927.534693,3331.959929
7,copper,Services,0.811872,0.811872
8,steel,Services,7.169211,133.504112
9,steel,Skin,7.083634,133.504112


In [28]:
# S-laag share per material_composite
df_slaag_composite_totals["share_slaag_composite"] = (
    df_slaag_composite_totals["kg_per_m2"] /
    df_slaag_composite_totals["total_kg_material"]
)

# Keep only relevant columns
df_slaag_composite_share = df_slaag_composite_totals[[
    "material_composite",
    "S-laag",
    "share_slaag_composite"
]]

# Sanity check (should sum to 1)
check = (
    df_slaag_composite_share
    .groupby("material_composite")["share_slaag_composite"]
    .sum()
)

print(check)
df_slaag_composite_share

material_composite
aluminium    1.0
concrete     1.0
copper       1.0
steel        1.0
Name: share_slaag_composite, dtype: float64


,material_composite,S-laag,share_slaag_composite
0,aluminium,Services,0.260205
1,aluminium,Skin,0.439397
2,aluminium,Space plan,0.300398
3,concrete,Services,0.000001
4,concrete,Skin,0.058667
5,concrete,Space plan,0.062709
6,concrete,Structure,0.878622
7,copper,Services,1.000000
8,steel,Services,0.053700
9,steel,Skin,0.053059


## 7. Sankey Diagram - S-laag Distribution

In [24]:
# Copy to avoid modifying original
df = df_slaag_composite_share.copy()

# Create node labels
materials = df["material_composite"].unique().tolist()
s_lagen = df["S-laag"].astype(str).unique().tolist()

labels = materials + s_lagen

# Create index mapping
label_to_index = {label: i for i, label in enumerate(labels)}

# Build Sankey links
sources = df["material_composite"].map(label_to_index)
targets = df["S-laag"].astype(str).map(label_to_index)
values = df["share_slaag_composite"] * 100

# Create Sankey figure
fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=20,
                thickness=20,
                line=dict(color="black", width=0.5),
                label=labels
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values))])

fig.update_layout(
    title_text="S-laag Distribution per Material Component",
    font_size=12,
    width=900,
    height=600)

fig.show()

# Save to HTML
fig.write_html("slaag_material_sankey.html")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 8. Load Additional Data for Material Intensities

In [ ]:
# Load building prognosis and material intensity data
data_building_prognostics = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_primos_building_prognosis")

data_building_material_intensities = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_material_intensities_build")

data_heat_pumps_material_intensities = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_material_intensities_heat_")

data_composite_material_share = pd.read_excel(
    "101125_building_input_data.xlsx", sheet_name="DATA_composite_material_share")

## 9. Extend Composite Material Share Data

In [ ]:
# Add aluminium and copper as pure materials
identity_materials = pd.DataFrame({
    "material_composite": ["aluminium", "copper"],
    "material_element": ["aluminium", "copper"],
    "share": [1.0, 1.0],
    "specifics": ["pure", "pure"]
})

data_composite_material_share_ext = pd.concat(
    [data_composite_material_share, identity_materials],
    ignore_index=True
)

## 10. Process Building Prognostics

In [ ]:
# Melt building_prognostics to long format
df_building_melt = data_building_prognostics.melt(
    id_vars="year",
    var_name="building_type",
    value_name="area_m2"
)

## 11. Calculate Material Mass per Building Type

In [ ]:
# Merge with material intensities
df_merged = pd.merge(
    df_building_melt,
    data_building_material_intensities,
    on="building_type",
    how="left"
)

# Compute total material mass per building type per year
for mat in ["concrete", "steel", "aluminium", "copper"]:
    df_merged[f"total_{mat}_kg"] = df_merged[mat] * df_merged["area_m2"]

# Melt to long format for materials
df_materials_long = df_merged.melt(
    id_vars=["year", "building_type"],
    value_vars=["total_concrete_kg", "total_steel_kg", "total_aluminium_kg", "total_copper_kg"],
    var_name="material_composite",
    value_name="total_kg"
)

# Clean column to match material_composite
df_materials_long["material_composite"] = (
    df_materials_long["material_composite"]
    .str.replace("total_", "", regex=False)
    .str.replace("_kg", "", regex=False)
)

## 12. Break Down Composites into Elements

In [ ]:
# Merge with composite-element share table
df_with_elements = df_materials_long.merge(
    data_composite_material_share_ext,
    on="material_composite",
    how="left"
)

# Compute element mass
df_with_elements["element_kg"] = df_with_elements["total_kg"] * df_with_elements["share"]

## 13. Aggregate Element Totals

In [ ]:
# Aggregate per year, building type, element
df_element_totals = (
    df_with_elements
    .groupby(["year", "building_type", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

# Pivot wide for plotting
df_element_pivot = df_element_totals.pivot_table(
    index="year",
    columns="material_element",
    values="element_kg",
    aggfunc="sum"
).fillna(0)

## 14. Calculate Cumulative Demand

In [ ]:
# Aggregate element mass per year, across all building types
df_element_totals_agg = (
    df_element_totals
    .groupby(["year", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

# Sort by year
df_element_totals_agg = df_element_totals_agg.sort_values("year")

# Compute cumulative sum
df_element_cum = (
    df_element_totals_agg
    .groupby("material_element", as_index=False)
    .apply(lambda x: x.assign(cum_kg=x["element_kg"].cumsum()))
    .reset_index(drop=True)
)

# Pivot to wide format
df_cum_pivot = df_element_cum.pivot(
    index="year",
    columns="material_element",
    values="cum_kg"
).fillna(0)

# Convert from kg to kilotonnes
df_cum_pivot_kt = df_cum_pivot / 1e6

## 15. Define Material Colors and Plotting Function

In [ ]:
# Material color scheme
material_colors = {
    "gravel": "#8B7D6B",        # brown-grey
    "sand": "#E6C27A",          # sand yellow
    "CEMIII": "#9E9E9E",        # cement grey
    "additives": "#6E6E6E",     # dark grey

    "silicon": "#F4A261",       # orange
    "manganese": "#2A9D8F",     # teal
    "chromium": "#264653",      # dark blue
    "molybdenum": "#6A4C93",    # purple
    "nickel": "#7CB342",        # green

    "aluminium": "#B0BEC5",     # light silver
    "copper": "#B87333"         # copper
}

def plot_stacked_cumulative(df, materials, title):
    """Plot stacked cumulative material demand over time"""
    df_plot = df[materials]

    colors = [material_colors[m] for m in materials]

    plt.figure(figsize=(14, 7))
    plt.stackplot(
        df_plot.index,
        df_plot.T.values,
        colors=colors,
        labels=materials,
        alpha=0.9
    )

    plt.title(title, fontsize=14)
    plt.ylabel("Cumulative Material Demand (kt)")
    plt.xlabel("Year")
    plt.xlim(2025, 2035)
    plt.legend(
        title="Material element",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.tight_layout()
    plt.show()

## 16. Visualize Cumulative Material Demand - All Materials

In [ ]:
# Define element order
element_order = [
    "gravel", "sand", "CEMIII", "additives",
    "silicon", "manganese", "chromium", "molybdenum", "nickel",
    "aluminium", "copper"
]

# Reorder columns
df_cum_pivot = df_cum_pivot.reindex(
    columns=[e for e in element_order if e in df_cum_pivot.columns]
)

# Plot all materials
materials_all = [m for m in element_order if m in df_cum_pivot_kt.columns]

plot_stacked_cumulative(
    df_cum_pivot_kt,
    materials_all,
    "Cumulative Material Demand Over Time — All Materials"
)

## 17. Visualize Sand and Metals

In [ ]:
# Sand and metals
materials_sand_metals = [
    "sand",
    "silicon", "manganese", "chromium", "molybdenum", "nickel",
    "aluminium", "copper"
]

materials_sand_metals = [m for m in materials_sand_metals if m in df_cum_pivot_kt.columns]

plot_stacked_cumulative(
    df_cum_pivot_kt,
    materials_sand_metals,
    "Cumulative Material Demand Over Time — Sand and Metals"
)

## 18. Visualize Metals Only

In [ ]:
# Metals only
materials_metals = [
    "silicon", "manganese", "chromium", "molybdenum", "nickel",
    "aluminium", "copper"
]

materials_metals = [m for m in materials_metals if m in df_cum_pivot_kt.columns]

plot_stacked_cumulative(
    df_cum_pivot_kt,
    materials_metals,
    "Cumulative Material Demand Over Time — Metals Only"
)

## 19. Calculate 2025-2035 Additions

In [ ]:
# Rebase to 2025
df_cum_rebased = df_cum_pivot_kt.sub(df_cum_pivot_kt.loc[2025])

# Extract only 2035
df_2035 = df_cum_rebased.loc[[2035]].T  # transpose to have materials as rows
df_2035 = df_2035.rename(columns={2035: "cum_add_2025_2035_kt"}).reset_index()
df_2035 = df_2035.rename(columns={"index": "material_element"})

df_2035.head()

## 20. Alternative Analysis - Element Calculations

In [ ]:
# Recalculate materials long format
df_materials_long = df_merged.melt(
    id_vars=["year", "building_type"],
    value_vars=[
        "total_concrete_kg",
        "total_steel_kg",
        "total_aluminium_kg",
        "total_copper_kg"
    ],
    var_name="material_composite",
    value_name="total_kg"
)

df_materials_long["material_composite"] = (
    df_materials_long["material_composite"]
    .str.replace("total_", "", regex=False)
    .str.replace("_kg", "", regex=False)
)

# Merge with shares
df_with_shares = df_materials_long.merge(
    data_composite_material_share_ext,
    on="material_composite",
    how="left"
)

df_with_shares["element_kg"] = (
    df_with_shares["total_kg"] * df_with_shares["share"]
)

# Aggregate
df_element_totals = (
    df_with_shares
    .groupby(["year", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

df_element_building = (
    df_with_shares
    .groupby(["year", "building_type", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

## 21. Bar Chart - Annual Material Demand

In [ ]:
# Group and pivot for bar chart
df_elements_grouped = (
    df_with_shares
    .groupby(["year", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

df_elements_pivot = df_elements_grouped.pivot(
    index="year",
    columns="material_element",
    values="element_kg"
).fillna(0)

# Reorder columns
df_elements_pivot = df_elements_pivot.reindex(
    columns=[e for e in element_order if e in df_elements_pivot.columns]
)

# Plot stacked bar chart
palette = sns.color_palette("tab20", n_colors=df_elements_pivot.shape[1])

df_elements_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 7),
    color=palette
)

plt.title("Total Material Demand per Year (by Material Element)", fontsize=14)
plt.ylabel("Total Material (kg)")
plt.xlabel("Year")
plt.legend(
    title="Material element",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## 22. Cumulative Demand - Stacked Area Plot

In [ ]:
# Calculate cumulative demand
df_elements_grouped = df_elements_grouped.sort_values("year")
df_elements_cum = (
    df_elements_grouped
    .groupby("material_element", as_index=False)
    .apply(lambda x: x.assign(cum_kg=x["element_kg"].cumsum()))
    .reset_index(drop=True)
)

# Pivot for plotting
df_elements_cum_pivot = df_elements_cum.pivot(
    index="year",
    columns="material_element",
    values="cum_kg"
).fillna(0)

df_elements_cum_pivot = df_elements_cum_pivot.reindex(
    columns=[e for e in element_order if e in df_elements_cum_pivot.columns]
)

# Plot stacked area
plt.figure(figsize=(14, 7))

colors = sns.color_palette("terrain", n_colors=len(df_elements_cum_pivot.columns))

plt.stackplot(
    df_elements_cum_pivot.index,
    df_elements_cum_pivot.T.values,
    colors=colors,
    labels=df_elements_cum_pivot.columns,
    alpha=0.9
)

plt.title("Cumulative Material Demand Over Time (by Material Element)", fontsize=14)
plt.ylabel("Cumulative Material Demand (kg)")
plt.xlabel("Year")

plt.legend(
    title="Material element",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## 23. Heat Pumps Analysis

In [ ]:
# Melt heat pump prognosis data
data_heat_pumps_prognostics_melt = data_building_prognostics.melt(
    id_vars="year",
    var_name="building_type",
    value_name="units"
)

# Merge with heat pump material intensities
df_hp_merged = pd.merge(
    data_heat_pumps_prognostics_melt,
    data_heat_pumps_material_intensities,
    on="building_type",
    how="left"
)

# Calculate total material for heat pumps
df_hp_merged["total_steel_kg"] = df_hp_merged["steel"] * df_hp_merged["units"]
df_hp_merged["total_aluminium_kg"] = df_hp_merged["aluminium"] * df_hp_merged["units"]
df_hp_merged["total_copper_kg"] = df_hp_merged["copper"] * df_hp_merged["units"]

## 24. Heat Pumps - Material Element Breakdown

In [ ]:
# Melt to long format
df_hp_materials_long = df_hp_merged.melt(
    id_vars=["year", "building_type"],
    value_vars=[
        "total_steel_kg",
        "total_aluminium_kg",
        "total_copper_kg"
    ],
    var_name="material_composite",
    value_name="total_kg"
)

df_hp_materials_long["material_composite"] = (
    df_hp_materials_long["material_composite"]
    .str.replace("total_", "", regex=False)
    .str.replace("_kg", "", regex=False)
)

# Merge with composite shares
df_hp_with_shares = df_hp_materials_long.merge(
    data_composite_material_share_ext,
    on="material_composite",
    how="left"
)

df_hp_with_shares["element_kg"] = (
    df_hp_with_shares["total_kg"] * df_hp_with_shares["share"]
)

# Aggregate
df_hp_element_totals = (
    df_hp_with_shares
    .groupby(["year", "material_element"], as_index=False)
    ["element_kg"]
    .sum()
)

df_hp_element_totals = df_hp_element_totals.sort_values("year")

# Cumulative sum
df_hp_elements_cum = (
    df_hp_element_totals
    .groupby("material_element", as_index=False)
    .apply(lambda x: x.assign(cum_kg=x["element_kg"].cumsum()))
    .reset_index(drop=True)
)

## 25. Heat Pumps - Cumulative Demand Visualization

In [ ]:
# Pivot to wide format for plotting
df_hp_cum_pivot = df_hp_elements_cum.pivot(
    index="year",
    columns="material_element",
    values="cum_kg"
).fillna(0)

# Reorder columns
df_hp_cum_pivot = df_hp_cum_pivot.reindex(
    columns=[e for e in element_order if e in df_hp_cum_pivot.columns]
)

# Plot
plt.figure(figsize=(14, 7))

colors = sns.color_palette("terrain", n_colors=len(df_hp_cum_pivot.columns))

plt.stackplot(
    df_hp_cum_pivot.index,
    df_hp_cum_pivot.T.values,
    colors=colors,
    labels=df_hp_cum_pivot.columns,
    alpha=0.9
)

plt.title("Cumulative Material Demand for Heat Pumps Over Time", fontsize=14)
plt.ylabel("Cumulative Material Demand (kg)")
plt.xlabel("Year")

plt.legend(
    title="Material element",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## 26. Export Results to Excel

In [ ]:
# Export S-laag composite share
df_slaag_composite_share.to_excel(
    "df_building_materials_by_slaag_composite.xlsx",
    index=False,
    sheet_name="S-laag_composite_split"
)

# Export building elements from intensities
df_with_elements.to_excel(
    "df_building_elements_from_intensities.xlsx",
    index=False,
    sheet_name="material_elements"
)

# Export element totals
df_element_totals.to_excel(
    "df_building_elements_totals.xlsx",
    index=False,
    sheet_name="material_elements"
)

# Export shares
df_with_shares.to_excel(
    "df_with_shares_material_elements.xlsx",
    sheet_name="material_elements",
    index=False
)

# Export cumulative elements
df_elements_cum.to_excel(
    "df_elements_cum.xlsx",
    sheet_name="material_elements",
    index=False
)

# Export heat pumps data
df_hp_with_shares.to_excel(
    "df_heat_pumps_material_elements.xlsx",
    sheet_name="material_elements",
    index=False
)

df_hp_elements_cum.to_excel(
    "df_heat_pumps_elements_cumulative.xlsx",
    sheet_name="material_elements",
    index=False
)

print("All results exported successfully!")